# **7. Validación**

In [1]:
# --- Manejo y análisis de datos ---
import pandas as pd
import numpy as np
import math
from collections import Counter
import time
import joblib

# --- Visualización ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Preprocesamiento ---
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

# --- División y validación ---
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold

# --- Modelos supervisados ---
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# --- Manejo de desbalanceo ---
from imblearn.over_sampling import SMOTENC
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# --- Métricas de evaluación ---
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    mean_absolute_error, mean_squared_error, r2_score
)

In [2]:
X_train = pd.read_csv('x_train_resampled.csv')
y_train = pd.read_csv('y_train_resampled.csv')["diabetes"]
X_test = pd.read_csv('X_test.csv')
y_test = pd.read_csv('y_test.csv')["diabetes"]

In [3]:
num_cols = X_train.select_dtypes(include="number").columns
cat_cols = X_train.select_dtypes(include="object").columns

In [4]:
# === Preprocesador ===
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
    ],
    remainder="drop"
)

cat_indices = [X_train.columns.get_loc(c) for c in cat_cols]

## **Modelo de `Regresión Logística L1 y L2`**

In [ ]:
pipe_logreg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=5000, solver='saga', class_weight='balanced', random_state=42))
])

param_grid_logreg = {
    'model__penalty': ['l1', 'l2'],
    'model__C': [0.01, 0.1, 1, 10]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_logreg = GridSearchCV(
    estimator=pipe_logreg,
    param_grid=param_grid_logreg,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2
)

start = time.time()
grid_logreg.fit(X_train, y_train)
end = time.time()


print("\n VALIDACIÓN CON DATOS BALANCEADOS COMPLETADA")
print(f"Tiempo total: {round(end - start, 2)} s")
print(f" Mejor AUC promedio (CV): {grid_logreg.best_score_:.3f}")
print(f" Mejores parámetros: {grid_logreg.best_params_}")

df_resultados_logreg = pd.DataFrame(grid_logreg.cv_results_)[
    ['mean_test_score', 'param_model__penalty', 'param_model__C']
].sort_values(by='mean_test_score', ascending=False)
display(df_resultados_logreg.head())




Fitting 5 folds for each of 8 candidates, totalling 40 fits

 VALIDACIÓN CON DATOS BALANCEADOS COMPLETADA
 Mejor AUC promedio (CV): 0.856
 Mejores parámetros: {'model__C': 10, 'model__penalty': 'l1'}


In [13]:
joblib.dump(grid_logreg.best_estimator_, "modelo_logistica.pkl")

['modelo_logistica.pkl']

## **Modelo de ``KNN_NeighborsClassifier``**

In [ ]:
pipe_knn = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', KNeighborsClassifier())
])

param_grid_knn = {
    'model__n_neighbors': [3, 5, 7, 9],
    'model__weights': ['uniform', 'distance'],
    'model__metric': ['minkowski', 'euclidean']
}

grid_knn = GridSearchCV(
    estimator=pipe_knn,
    param_grid=param_grid_knn,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2
)

start = time.time()
grid_knn.fit(X_train, y_train)
end = time.time()

print("\nVALIDACIÓN COMPLETADA - KNN")
print(f"Tiempo: {round(end - start, 2)} s")
print(f" Mejor AUC (CV): {grid_knn.best_score_:.3f}")
print(f" Mejores parámetros: {grid_knn.best_params_}")

df_resultados_knn = pd.DataFrame(grid_knn.cv_results_)[
    ['mean_test_score', 'param_model__n_neighbors', 'param_model__weights', 'param_model__metric']
].sort_values(by='mean_test_score', ascending=False)
display(df_resultados_knn.head())


Fitting 5 folds for each of 16 candidates, totalling 80 fits

VALIDACIÓN COMPLETADA - KNN
Tiempo: 1701.12 s
 Mejor AUC (CV): 0.917
 Mejores parámetros: {'model__metric': 'minkowski', 'model__n_neighbors': 9, 'model__weights': 'distance'}


,mean_test_score,param_model__n_neighbors,param_model__weights,param_model__metric
7,0.917363,9,distance,minkowski
15,0.917363,9,distance,euclidean
13,0.915518,7,distance,euclidean
5,0.915518,7,distance,minkowski
3,0.910312,5,distance,minkowski


In [14]:
joblib.dump(grid_knn.best_estimator_, 'modelo_knn.pkl')

['modelo_knn.pkl']

## **Modelo de ``Naive Bayes``**

In [ ]:
pipe_nb = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', GaussianNB())
])

param_grid_nb = {
    'model__var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]
}

grid_nb = GridSearchCV(
    estimator=pipe_nb,
    param_grid=param_grid_nb,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2
)

start = time.time()
grid_nb.fit(X_train, y_train)
end = time.time()

print("\nVALIDACIÓN COMPLETADA - NAIVE BAYES")
print(f"Tiempo: {round(end - start, 2)} s")
print(f"Mejor AUC (CV): {grid_nb.best_score_:.3f}")
print(f"Mejores parámetros: {grid_nb.best_params_}")

df_resultados_nb = pd.DataFrame(grid_nb.cv_results_)[
    ['mean_test_score', 'param_model__var_smoothing']
].sort_values(by='mean_test_score', ascending=False)
display(df_resultados_nb.head())


Fitting 5 folds for each of 4 candidates, totalling 20 fits


c:\Users\franc\anaconda3\envs\ml_env\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



VALIDACIÓN COMPLETADA - NAIVE BAYES
Tiempo: 14.55 s
Mejor AUC (CV): 0.788
Mejores parámetros: {'model__var_smoothing': 1e-06}


,mean_test_score,param_model__var_smoothing
3,0.788252,1.000000e-06
2,0.788251,1.000000e-07
1,0.788251,1.000000e-08
0,0.788251,1.000000e-09


In [15]:
joblib.dump(grid_nb.best_estimator_, 'modelo_nb.pkl')

['modelo_nb.pkl']

## **Modelo de ``DecisionTreeClassifier``**

In [ ]:

pipe_tree = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', DecisionTreeClassifier(random_state=42))
])

param_grid_tree = {
    'model__max_depth': [3, 5, 7, 10],
    'model__criterion': ['gini', 'entropy']
}

grid_tree = GridSearchCV(
    estimator=pipe_tree,
    param_grid=param_grid_tree,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2
)

start = time.time()
grid_tree.fit(X_train, y_train)
end = time.time()

print("\n VALIDACIÓN COMPLETADA - DECISION TREE")
print(f"Tiempo: {round(end - start, 2)} s")
print(f" Mejor AUC (CV): {grid_tree.best_score_:.3f}")
print(f" Mejores parámetros: {grid_tree.best_params_}")

df_resultados_tree = pd.DataFrame(grid_tree.cv_results_)[
    ['mean_test_score', 'param_model__max_depth', 'param_model__criterion']
].sort_values(by='mean_test_score', ascending=False)
display(df_resultados_tree.head())


Fitting 5 folds for each of 8 candidates, totalling 40 fits

 VALIDACIÓN COMPLETADA - DECISION TREE
Tiempo: 34.64 s
 Mejor AUC (CV): 0.855
 Mejores parámetros: {'model__criterion': 'gini', 'model__max_depth': 10}


,mean_test_score,param_model__max_depth,param_model__criterion
3,0.855361,10,gini
7,0.853426,10,entropy
2,0.811810,7,gini
6,0.811081,7,entropy
5,0.783702,5,entropy


['modelo_tree.pkl']

In [16]:
joblib.dump(grid_tree.best_estimator_, 'modelo_tree.pkl')

['modelo_tree.pkl']

## **Modelo de ``RandomForestClassifier``**

In [ ]:
pipe_rf = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [5, 10, None],
    'model__min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(
    estimator=pipe_rf,
    param_grid=param_grid_rf,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2
)

start = time.time()
grid_rf.fit(X_train, y_train)
end = time.time()

print("\n VALIDACIÓN COMPLETADA - RANDOM FOREST")
print(f"Tiempo: {round(end - start, 2)} s")
print(f" Mejor AUC (CV): {grid_rf.best_score_:.3f}")
print(f" Mejores parámetros: {grid_rf.best_params_}")

df_resultados_rf = pd.DataFrame(grid_rf.cv_results_)[
    ['mean_test_score', 'param_model__n_estimators', 'param_model__max_depth', 'param_model__min_samples_split']
].sort_values(by='mean_test_score', ascending=False)
display(df_resultados_rf.head())



Fitting 5 folds for each of 18 candidates, totalling 90 fits

 VALIDACIÓN COMPLETADA - RANDOM FOREST
Tiempo: 3347.74 s
 Mejor AUC (CV): 0.969
 Mejores parámetros: {'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 300}


,mean_test_score,param_model__n_estimators,param_model__max_depth,param_model__min_samples_split
14,0.968545,300,None,2
13,0.968209,200,None,2
12,0.967534,100,None,2
17,0.965744,300,None,5
16,0.965402,200,None,5


['modelo_rf_cv.pkl']

In [18]:
joblib.dump(grid_rf.best_estimator_, 'modelo_rf.pkl')

['modelo_rf.pkl']

## **Modelo de ``XGBClassifier``**

In [ ]:
pipe_xgb = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', XGBClassifier(
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    ))
])

param_grid_xgb = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [4, 6, 8],
    'model__learning_rate': [0.01, 0.05, 0.1]
}

grid_xgb = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=param_grid_xgb,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2
)

start = time.time()
grid_xgb.fit(X_train, y_train)
end = time.time()

print("\n VALIDACIÓN COMPLETADA - XGBOOST")
print(f"Tiempo: {round(end - start, 2)} s")
print(f" Mejor AUC (CV): {grid_xgb.best_score_:.3f}")
print(f" Mejores parámetros: {grid_xgb.best_params_}")

df_resultados_xgb = pd.DataFrame(grid_xgb.cv_results_)[
    ['mean_test_score', 'param_model__n_estimators', 'param_model__max_depth', 'param_model__learning_rate']
].sort_values(by='mean_test_score', ascending=False)
display(df_resultados_xgb.head())

joblib.dump(grid_xgb.best_estimator_, 'modelo_xgb_cv.pkl')

Fitting 5 folds for each of 27 candidates, totalling 135 fits


c:\Users\franc\anaconda3\envs\ml_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [00:04:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



 VALIDACIÓN COMPLETADA - XGBOOST
Tiempo: 271.58 s
 Mejor AUC (CV): 0.966
 Mejores parámetros: {'model__learning_rate': 0.1, 'model__max_depth': 8, 'model__n_estimators': 300}


,mean_test_score,param_model__n_estimators,param_model__max_depth,param_model__learning_rate
26,0.966129,300,8,0.10
25,0.965369,200,8,0.10
23,0.964820,300,6,0.10
17,0.964584,300,8,0.05
22,0.963561,200,6,0.10


['modelo_xgb_cv.pkl']

In [19]:
joblib.dump(grid_xgb.best_estimator_, 'modelo_xgb.pkl')

['modelo_xgb.pkl']

## **Modelo de ``LinearSVC``**

In [ ]:
pipe_svm = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', CalibratedClassifierCV(
        estimator=LinearSVC(max_iter=5000, class_weight='balanced', random_state=42),
        cv=3
    ))
])

param_grid_svm = {
    'model__estimator__C': [0.01, 0.1, 1, 10]   #  parámetro actualizado
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_svm = GridSearchCV(
    estimator=pipe_svm,
    param_grid=param_grid_svm,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=2
)

start = time.time()
grid_svm.fit(X_train, y_train)
end = time.time()

print("\n VALIDACIÓN COMPLETADA - LINEAR SVC (Calibrado)")
print(f"Tiempo total: {round(end - start, 2)} s")
print(f" Mejor AUC promedio (CV): {grid_svm.best_score_:.3f}")
print(f" Mejores parámetros: {grid_svm.best_params_}")

df_resultados_svm = pd.DataFrame(grid_svm.cv_results_)[
    ['mean_test_score', 'param_model__estimator__C']
].sort_values(by='mean_test_score', ascending=False)
display(df_resultados_svm.head())



Fitting 5 folds for each of 4 candidates, totalling 20 fits

 VALIDACIÓN COMPLETADA - LINEAR SVC (Calibrado)
Tiempo total: 54.43 s
 Mejor AUC promedio (CV): 0.856
 Mejores parámetros: {'model__estimator__C': 10}


,mean_test_score,param_model__estimator__C
3,0.855719,10.00
2,0.855715,1.00
1,0.855680,0.10
0,0.855307,0.01


In [20]:

joblib.dump(grid_svm.best_estimator_, 'modelo_svm.pkl')

['modelo_svm.pkl']